# Installing Packages

The Required Libraries for the current process is loaded into the system by executing the following commands in a seperate python virtual environment's terminal:

```bash
pip install numpy
pip install pandas
pip install Pillow
pip install torch 
pip install torchvision 
pip install scikit-learn
```

The rest of the Libraries are already included in the python installation by default

# Importing Libraries

In [24]:
from PIL import Image
import pandas as pd
import numpy as np
import os

In [25]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torch

# Load the Dataset

In [26]:
DATASET_PATH = "Dataset"
IMAGE_PATH = os.path.join(DATASET_PATH, "Images")

In [27]:
df = pd.read_csv(os.path.join(DATASET_PATH, "02. cleaned_dataset.csv"))

In [28]:
print("Dataset size:", len(df))
df.head()

Dataset size: 14999


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,...,Fibrosis,Hernia,Infiltration,Mass,No Finding,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax,split
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,...,0,0,0,0,0,0,0,0,0,train_val
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,...,0,0,0,0,0,0,0,0,0,train_val
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,...,0,0,0,0,0,0,0,0,0,train_val
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,...,0,0,0,0,1,0,0,0,0,train_val
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,...,0,1,0,0,0,0,0,0,0,test


# Defining Classes

In [29]:
all_labels = set()
for labels_str in df["Finding Labels"]:
    for disease in labels_str.split("|"):
        all_labels.add(disease)

diseases = sorted(all_labels)

In [30]:
diseases

['Atelectasis',
 'Cardiomegaly',
 'Consolidation',
 'Edema',
 'Effusion',
 'Emphysema',
 'Fibrosis',
 'Hernia',
 'Infiltration',
 'Mass',
 'No Finding',
 'Nodule',
 'Pleural_Thickening',
 'Pneumonia',
 'Pneumothorax']

# Train/Validation Split

In [31]:
train_val_df = df[df["split"] == "train_val"].copy()
test_df = df[df["split"] == "test"].copy()

In [32]:
patients = train_val_df["Patient ID"].unique()

In [33]:
train_patients, val_patients = train_test_split(
    patients,
    test_size=0.2,
    random_state=42
)

In [34]:
train_df = train_val_df[train_val_df["Patient ID"].isin(train_patients)]
val_df = train_val_df[train_val_df["Patient ID"].isin(val_patients)]

In [35]:
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 9989
Val: 2551
Test: 2459


# Check for Image Color Channel

In [36]:
img_path = os.path.join(IMAGE_PATH, train_df.iloc[0]["Image Index"])
img = Image.open(img_path)

print("Image mode:", img.mode)
print("Image size:", img.size)

Image mode: L
Image size: (1024, 1024)


# Image Transformations

First, we define the Transform Pipeline Object, which will help us transform our Images in our dataset. Here the Pipelines are defined in the following ways:
1. **Train Transform Pipeline**: It transforms the image in the following way:
    - Resizes the Image
    - Randomly flips some images Horizontally
    - Randomly rotates some images within ±10°
    - Randomly adjusts brightness and contrast within ±10% for all images
    - Normalizes Tensor

2. **Validation Transform Pipeline**: It transforms the image in the following ways:
    - Resizes the Image
    - Normalizes Tensor

3. **Test Transform Pipeline**: It transforms the image in the following ways:
    - Resizes the Image
    - Normalizes Tensor

In [37]:
# Slight Random Augmentations
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [38]:
# No Augmentations
val_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [39]:
# No Augmentations
test_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Dataset Class

We'd create a Dataset Class named `ChestXrayDataset` to handle the Dataset effectively. The reason that we need a Dataset class is that we can implement *Lazy Loading* in our Model Training, which makes model training **Scalable**. It is also used to attach image Metadata to the images and perform Image Transformations. We also create three objects out of our `ChestXrayDataset` class named following:
- `train_dataset`: This object is our Training Dataset processed with `train_transforms`
- `val_dataset`: This object is our Validation Dataset processed with `val_transforms`
- `test_dataset`: This object is our Test Dataset preprocessed with `test_transforms`

In [40]:
class ChestXrayDataset(Dataset):

    def __init__(self, dataframe, image_dir, label_cols, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.label_cols = label_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row["Image Index"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)
        labels = torch.tensor(
            row[self.label_cols].values.astype(np.float32)
        )

        return image, labels

In [41]:
train_dataset = ChestXrayDataset(
    train_df,
    IMAGE_PATH,
    diseases,
    transform=train_transforms
)

val_dataset = ChestXrayDataset(
    val_df,
    IMAGE_PATH,
    diseases,
    transform=val_transforms
)

test_dataset = ChestXrayDataset(
    test_df,
    IMAGE_PATH,
    diseases,
    transform=test_transforms
)

# Creating DataLoaders

Now that we have our Dataset Objects, we can now proceed to create the `DataLoader` to our Dataset, which allows us to achieve *Lazy Loading* at our Model Training phase. Lazy Loading helps the model training to be efficient and scalable without over-utilizing the resources.


## Purpose of DataLoaders

- Traditional Procedure
    ```bash
    Dataset (approx. 15000) -> RAM -> Process
    ```

- DataLoaders
    ```bash
    Dataset (00 - 32) -> RAM -> Process
    Dataset (32 - 64) -> RAM -> Process
    ...
    ```
    , where the Batch size is determined by the `batch_size` value of the DataLoader

In [42]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

# Test Pipeline

Test the Pipeline, just to be on the safer side

In [43]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32, 15])


# Export Splits

In [45]:
train_df.to_csv("Dataset/03. train_df.csv", index=False)
val_df.to_csv("Dataset/03. val_df.csv", index=False)
test_df.to_csv("Dataset/03. test_df.csv", index=False)